# Qwen 2.5 3B — Compound TOE-to-CPE Extraction (Colab / Kaggle Quickstart)

Run this notebook on a **free Google Colab T4 GPU (15 GB VRAM)** or **Kaggle T4/P100 GPU (16 GB VRAM)** — no local GPU required.

> **Before running:** In Colab, go to **Runtime → Change runtime type → T4 GPU**.

In [1]:
# 1. Clone the GitHub repository and install dependencies
REPO_URL = "https://github.com/m-mirkan/peft-cpe-extraction-pipeline.git"
BRANCH = "feat/codebook-data-curation"

import os
if not os.path.exists("repo"):
    !git clone -b {BRANCH} {REPO_URL} repo
%cd repo
# Remove Colab's outdated preinstalled torchao (conflicts with latest peft)
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt gdown

import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Cloning into 'repo'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 55 (delta 19), reused 32 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 2.47 MiB | 14.07 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/repo
CUDA available: True | GPU: Tesla T4


In [2]:
# 2. Download the pre-trained baseline LoRA adapter from Google Drive
import os, zipfile

LORA_GDRIVE_ID = "1eKyqTdDT36zPEpi6y62kZg9V8vTmJVaX"
LORA_DIR = "checkpoints/qwen25_3b_cpe_lora"
os.makedirs(LORA_DIR, exist_ok=True)

!gdown --id {LORA_GDRIVE_ID} -O lora_adapter.zip

with zipfile.ZipFile("lora_adapter.zip", "r") as z:
    z.extractall(LORA_DIR)

# Handle case where zip contains a single nested folder
entries = os.listdir(LORA_DIR)
if len(entries) == 1 and os.path.isdir(os.path.join(LORA_DIR, entries[0])):
    nested = os.path.join(LORA_DIR, entries[0])
    for f in os.listdir(nested):
        os.rename(os.path.join(nested, f), os.path.join(LORA_DIR, f))
    os.rmdir(nested)

print("LoRA adapter files:", os.listdir(LORA_DIR))

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1eKyqTdDT36zPEpi6y62kZg9V8vTmJVaX
From (redirected): https://drive.google.com/uc?id=1eKyqTdDT36zPEpi6y62kZg9V8vTmJVaX&confirm=t&uuid=38930ba8-d41e-4668-a7aa-65080aab9e13
To: /content/repo/lora_adapter.zip
100% 113M/113M [00:01<00:00, 68.4MB/s]
LoRA adapter files: ['chat_template.jinja', 'tokenizer.json', 'adapter_model.safetensors', 'README.md', 'tokenizer_config.json', 'adapter_config.json']


In [ ]:
# 3. Load Base Qwen2.5-3B + Switchable LoRA Adapter (auto-downloads base model from HF)
import os, sys
if os.path.exists("repo") and not os.path.exists("src"):
    os.chdir("repo")
if "src" not in sys.path:
    sys.path.insert(0, "src")

from cpe_resolver import CPEDictionaryResolver
from interactive_cpe_shell import load_switchable_model, run_single_extraction, print_result

resolver = CPEDictionaryResolver()
hf_model, outlines_model, has_lora = load_switchable_model(
    "Qwen/Qwen2.5-3B-Instruct",
    "checkpoints/qwen25_3b_cpe_lora",
)

In [ ]:
# 4. Compare Baseline (Zero-Shot) vs. Fine-Tuned LoRA on any arbitrary TOE string!
test_text = "Fuji Xerox DocuCentre-IV C2265/C2263 Series Controller Software for Asia Pacific Version: Controller ROM Ver. 1.101.12"
attach_fw_to_models = False  # Set True for model-bound `_firmware` CPEs

b_parsed, b_cpes, b_time = run_single_extraction(
    hf_model, outlines_model, resolver, test_text,
    use_lora=False, has_lora=has_lora, attach_fw=attach_fw_to_models
)
print_result("BASELINE (Zero-Shot Qwen2.5-3B)", test_text, b_parsed, b_cpes, b_time, attach_fw=attach_fw_to_models)

ft_parsed, ft_cpes, ft_time = run_single_extraction(
    hf_model, outlines_model, resolver, test_text,
    use_lora=True, has_lora=has_lora, attach_fw=attach_fw_to_models
)
print_result("FINE-TUNED (QLoRA)", test_text, ft_parsed, ft_cpes, ft_time, attach_fw=attach_fw_to_models)

In [4]:
# 5. (Optional) Run batch evaluation on the benchmark files
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora --input_file data/benchmarks/names.json --offset 0 --limit 50 --output_file outputs/names_part1.json

[Infer] Loading base model `Qwen/Qwen2.5-3B-Instruct` on cuda (torch.bfloat16)...
config.json: 100% 661/661 [00:00<00:00, 3.02MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 86.1MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 25.1M/6.17G [00:01<04:38, 22.1MB/s, 26.6kB/s  ]
Reconstructing (incomplete total...):  31% 1.91G/6.17G [00:24<01:36, 44.3MB/s,  122MB/s  ]

Reconstructing (incomplete total...):  91% 5.59G/6.17G [00:54<00:06, 94.3MB/s,  106MB/s  ]
Reconstructing (incomplete total...):  97% 6.00G/6.17G [00:56<00:01, 131MB/s, 94.3MB/s  ] 
Reconstructing (incomplete total...): 100% 6.17G/6.17G [00:56<00:00, 146MB/s,  131MB/s  ]

Fet

In [5]:
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora --input_file data/benchmarks/names.json --offset 50 --limit 50 --output_file outputs/names_part2.json

[Infer] Loading base model `Qwen/Qwen2.5-3B-Instruct` on cuda (torch.bfloat16)...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:12<00:00, 33.39it/s]
[Infer] Attaching fine-tuned LoRA adapter from `checkpoints/qwen25_3b_cpe_lora`...

[51] INPUT: Infineon Security Controller M7892 B11 with optional RSA2048/4096 v1.02.013, EC v1.02.013, SHA-2 v1.01, SCL v2.02.012, Base v1.02.013, and Toolbox v1.02.013 libraries and with specific IC dedicated software (firmware)
---------------------------------------------------------------------------
  h:infineon:security_controller_m7892_b11:-         [explicit_vendor]
  a:infineon:rsa2048_4096:1.02.013                   [clause_inheritance]
  a:infineon:ec:1.02.013                             [clause_inheritance]
  a:infineon:sha-2:1.01                              [clause_inheritance]
  a:schneider-electric:scl:2.02.012                  [cpe_prefix(scl)]
  a:infineon:base:1.02.013                  

In [ ]:
# 6. (Optional) Train your own improved QLoRA adapter on T4/P100 GPU
# Note: Restart runtime first if you already loaded the inference model above to free VRAM.
# !python src/train_qwen.py \
#   --train_file data/synthetic/synthetic_train.jsonl \
#   --val_file data/synthetic/synthetic_val.jsonl \
#   --output_dir checkpoints/my_improved_lora \
#   --epochs 1 --batch_size 2 --grad_accum 8